##### Level 1 - Agentic AI Collections Agent (Basic Version)
##### Author: Oluwafisayomi Harrison Ajiboye
##### Date: 04/11/2026

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

print("✅ Libraries imported successfully")
print("Starting Level 1 Collections Agent Prototype")

✅ Libraries imported successfully
Starting Level 1 Collections Agent Prototype


In [12]:
# Load the cleaned dataset with the improved target
df = pd.read_excel("/Users\HP\Projects\Gen AI Powered Data Analytics\Cleaned_Delinquency_Dataset_with_New_Target.xlsx")

print(f"✅ Dataset loaded successfully!")
print(f"Total customers: {len(df)}")
print(f"Columns available: {list(df.columns)}")
print(f"\nTarget distribution:\n{df['Delinquent_Account_New'].value_counts()}")

<>:2: SyntaxWarning: invalid escape sequence '\H'
<>:2: SyntaxWarning: invalid escape sequence '\H'
C:\Users\HP\AppData\Local\Temp\ipykernel_35664\1929292457.py:2: SyntaxWarning: invalid escape sequence '\H'
  df = pd.read_excel("/Users\HP\Projects\Gen AI Powered Data Analytics\Cleaned_Delinquency_Dataset_with_New_Target.xlsx")


✅ Dataset loaded successfully!
Total customers: 500
Columns available: ['Customer_ID', 'Age', 'Income', 'Credit_Score', 'Credit_Utilization', 'Missed_Payments', 'Delinquent_Account', 'Loan_Balance', 'Debt_to_Income_Ratio', 'Employment_Status', 'Account_Tenure', 'Credit_Card_Type', 'Location', 'Month_1', 'Month_2', 'Month_3', 'Month_4', 'Month_5', 'Month_6', 'Delinquent_Account_New']

Target distribution:
Delinquent_Account_New
1    254
0    246
Name: count, dtype: int64


### - Risk Scoring Function

In [14]:
# Code Block 3: Simple Risk Scoring Function
def calculate_risk_score(row):
    """
    Calculate a simple risk score (0 to 100) based on key features.
    Higher score = higher risk of delinquency.
    """
    score = 0
    
    # Strongest factor: Missed Payments
    if row['Missed_Payments'] >= 4:
        score += 50
    elif row['Missed_Payments'] >= 2:
        score += 25
    
    # Credit Utilization
    if row['Credit_Utilization'] > 0.70:
        score += 30
    elif row['Credit_Utilization'] > 0.50:
        score += 15
    
    # Credit Score (lower = higher risk)
    if row['Credit_Score'] < 500:
        score += 15
    elif row['Credit_Score'] < 600:
        score += 8
    
    # Cap the score at 100
    return min(score, 100)

# Apply the risk scoring to the entire dataset
df['Risk_Score'] = df.apply(calculate_risk_score, axis=1)

print("✅ Risk scoring function created and applied")
print(df[['Customer_ID', 'Missed_Payments', 'Credit_Utilization', 'Risk_Score']].head(8))

✅ Risk scoring function created and applied
  Customer_ID  Missed_Payments  Credit_Utilization  Risk_Score
0    CUST0001                3            0.390502          40
1    CUST0002                6            0.312444          65
2    CUST0003                0            0.359930           8
3    CUST0004                3            0.371400          40
4    CUST0005                2            0.234716          40
5    CUST0006                6            0.650540          65
6    CUST0007                3            0.390581          40
7    CUST0008                5            0.532715          80


### - Action Recommendation Logic

In [15]:
# Code Block 4: Action Recommendation Function
def recommend_action(row):
    """
    Recommends the best collection action based on risk score and customer profile.
    """
    score = row['Risk_Score']
    missed = row['Missed_Payments']
    utilization = row['Credit_Utilization']
    
    if score >= 70 or missed >= 5:
        return "ESCALATE"          # Immediate human intervention needed
    elif score >= 50 or missed >= 4 or utilization > 0.70:
        return "PAYMENT_PLAN"      # Offer structured payment plan + reminder
    elif score >= 30 or missed >= 2:
        return "SMS_REMINDER"      # Gentle automated reminder
    else:
        return "MONITOR"           # Low risk - just monitor

# Apply the recommendation to the dataset
df['Recommended_Action'] = df.apply(recommend_action, axis=1)

print("✅ Action recommendation function applied")
print("\nAction Distribution:")
print(df['Recommended_Action'].value_counts())

# Preview
print("\nSample recommendations:")
display(df[['Customer_ID', 'Missed_Payments', 'Credit_Utilization', 
            'Risk_Score', 'Recommended_Action']].head(10))

✅ Action recommendation function applied

Action Distribution:
Recommended_Action
ESCALATE        169
SMS_REMINDER    132
PAYMENT_PLAN    100
MONITOR          99
Name: count, dtype: int64

Sample recommendations:


,Customer_ID,Missed_Payments,Credit_Utilization,Risk_Score,Recommended_Action
0,CUST0001,3,0.390502,40,SMS_REMINDER
1,CUST0002,6,0.312444,65,ESCALATE
2,CUST0003,0,0.359930,8,MONITOR
3,CUST0004,3,0.371400,40,SMS_REMINDER
4,CUST0005,2,0.234716,40,SMS_REMINDER
5,CUST0006,6,0.650540,65,ESCALATE
6,CUST0007,3,0.390581,40,SMS_REMINDER
7,CUST0008,5,0.532715,80,ESCALATE
8,CUST0009,5,0.413035,65,ESCALATE
9,CUST0010,4,0.361824,50,PAYMENT_PLAN


### Guardrails & Ethical Checks

In [16]:
# Code Block 5: Basic Guardrails (Ethical & Fairness Checks)
def apply_guardrails(row):
    """
    Applies basic ethical guardrails before finalizing action.
    """
    action = row['Recommended_Action']
    age = row['Age']
    employment = str(row['Employment_Status']).lower()
    
    # Guardrail 1: Vulnerable customers (senior citizens or unemployed)
    if age >= 65 or 'unemployed' in employment or 'retired' in employment:
        if action == "ESCALATE":
            return "PAYMENT_PLAN"   # Soften aggressive action for vulnerable groups
        elif action == "PAYMENT_PLAN":
            return "SMS_REMINDER"   # Even softer approach
    
    # Guardrail 2: First-time offenders
    if row['Missed_Payments'] == 1:
        return "SMS_REMINDER"
    
    return action

# Apply guardrails
df['Final_Action'] = df.apply(apply_guardrails, axis=1)

print("✅ Guardrails applied successfully")
print("\nFinal Action Distribution after Guardrails:")
print(df['Final_Action'].value_counts())

✅ Guardrails applied successfully

Final Action Distribution after Guardrails:
Final_Action
SMS_REMINDER    223
PAYMENT_PLAN    130
ESCALATE         89
MONITOR          58
Name: count, dtype: int64


This function protects vulnerable customers (elderly, unemployed, retired) from harsh actions.
It also softens actions for customers with only 1 missed payment.

### - Main Agent Function

In [17]:
# Code Block 6: The Core Agent Function
def collections_agent(customer_row):
    """
    Main function that runs the full Level 1 Agent for one customer.
    """
    # Step 1: Calculate Risk
    risk_score = calculate_risk_score(customer_row)
    
    # Step 2: Recommend initial action
    action = recommend_action(customer_row)
    
    # Step 3: Apply guardrails
    final_action = apply_guardrails(customer_row)
    
    # Step 4: Create log entry
    log = {
        'Customer_ID': customer_row['Customer_ID'],
        'Risk_Score': risk_score,
        'Initial_Action': action,
        'Final_Action': final_action,
        'Timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
    
    return log, final_action

print("✅ Main Collections Agent function created")

✅ Main Collections Agent function created


### - Test the Agent on Sample Customers

In [18]:
# Code Block 7: Test the Agent
print("🚀 Testing Collections Agent on Sample Customers\n")

sample_customers = df.sample(5, random_state=42)

results = []

for idx, customer in sample_customers.iterrows():
    log, action = collections_agent(customer)
    results.append(log)
    print(f"Customer {customer['Customer_ID']}: Risk Score = {log['Risk_Score']}, "
          f"Action = {action}")

# Convert results to DataFrame for easy viewing
results_df = pd.DataFrame(results)
display(results_df)

🚀 Testing Collections Agent on Sample Customers

Customer CUST0362: Risk Score = 55, Action = PAYMENT_PLAN
Customer CUST0074: Risk Score = 70, Action = ESCALATE
Customer CUST0375: Risk Score = 30, Action = SMS_REMINDER
Customer CUST0156: Risk Score = 30, Action = PAYMENT_PLAN
Customer CUST0105: Risk Score = 40, Action = SMS_REMINDER


,Customer_ID,Risk_Score,Initial_Action,Final_Action,Timestamp
0,CUST0362,55,PAYMENT_PLAN,PAYMENT_PLAN,2026-04-11 22:30:10
1,CUST0074,70,ESCALATE,ESCALATE,2026-04-11 22:30:10
2,CUST0375,30,PAYMENT_PLAN,SMS_REMINDER,2026-04-11 22:30:10
3,CUST0156,30,PAYMENT_PLAN,PAYMENT_PLAN,2026-04-11 22:30:10
4,CUST0105,40,SMS_REMINDER,SMS_REMINDER,2026-04-11 22:30:10


### - Logging System (Save Decisions)

In [19]:
# Code Block 8: Logging System - Save all agent decisions to a file
import os
from datetime import datetime

# Create logs folder if it doesn't exist
os.makedirs('logs', exist_ok=True)

def log_agent_decision(log_data):
    """Save agent decision to a CSV log file"""
    log_df = pd.DataFrame([log_data])
    
    log_file = 'logs/collections_agent_log.csv'
    
    # Append to log file (create if doesn't exist)
    if os.path.exists(log_file):
        log_df.to_csv(log_file, mode='a', header=False, index=False)
    else:
        log_df.to_csv(log_file, index=False)
    
    print(f"✅ Decision logged for {log_data['Customer_ID']}")

print("✅ Logging system ready")

✅ Logging system ready


### - Run Agent on Multiple Customers (Batch Processing)

In [20]:
# Code Block 9: Batch Processing - Run agent on many customers
print("🚀 Running Collections Agent on 20 sample customers...\n")

sample_df = df.sample(20, random_state=42).copy()
logs = []

for idx, customer in sample_df.iterrows():
    log, final_action = collections_agent(customer)
    logs.append(log)
    log_agent_decision(log)   # Save to log file
    
    print(f"✅ {customer['Customer_ID']:10} | Risk: {log['Risk_Score']:3} | Action: {final_action}")

print("\n🎉 Batch processing completed!")
print(f"Total decisions logged: {len(logs)}")

🚀 Running Collections Agent on 20 sample customers...

✅ Decision logged for CUST0362
✅ CUST0362   | Risk:  55 | Action: PAYMENT_PLAN
✅ Decision logged for CUST0074
✅ CUST0074   | Risk:  70 | Action: ESCALATE
✅ Decision logged for CUST0375
✅ CUST0375   | Risk:  30 | Action: SMS_REMINDER
✅ Decision logged for CUST0156
✅ CUST0156   | Risk:  30 | Action: PAYMENT_PLAN
✅ Decision logged for CUST0105
✅ CUST0105   | Risk:  40 | Action: SMS_REMINDER
✅ Decision logged for CUST0395
✅ CUST0395   | Risk:  48 | Action: SMS_REMINDER
✅ Decision logged for CUST0378
✅ CUST0378   | Risk:  48 | Action: SMS_REMINDER
✅ Decision logged for CUST0125
✅ CUST0125   | Risk:  15 | Action: MONITOR
✅ Decision logged for CUST0069
✅ CUST0069   | Risk:  50 | Action: PAYMENT_PLAN
✅ Decision logged for CUST0451
✅ CUST0451   | Risk:  58 | Action: PAYMENT_PLAN
✅ Decision logged for CUST0010
✅ CUST0010   | Risk:  50 | Action: PAYMENT_PLAN
✅ Decision logged for CUST0195
✅ CUST0195   | Risk:  40 | Action: SMS_REMINDER
✅ Deci

### - Final Summary & Agent Status

In [21]:
# Code Block 10: Final Summary
print("="*60)
print("LEVEL 1 AGENTIC AI COLLECTIONS AGENT - SUMMARY")
print("="*60)
print(f"Total customers processed : {len(df)}")
print(f"High Risk (ESCALATE)      : {len(df[df['Final_Action'] == 'ESCALATE'])}")
print(f"Medium Risk (PAYMENT_PLAN): {len(df[df['Final_Action'] == 'PAYMENT_PLAN'])}")
print(f"Low Risk (SMS_REMINDER)   : {len(df[df['Final_Action'] == 'SMS_REMINDER'])}")
print(f"Monitor Only              : {len(df[df['Final_Action'] == 'MONITOR'])}")
print(f"\nLog file saved at: logs/collections_agent_log.csv")
print("\n✅ Level 1 Basic Agent is complete and working!")
print("This agent combines rule-based logic with simple reasoning and ethical guardrails.")

LEVEL 1 AGENTIC AI COLLECTIONS AGENT - SUMMARY
Total customers processed : 500
High Risk (ESCALATE)      : 89
Medium Risk (PAYMENT_PLAN): 130
Low Risk (SMS_REMINDER)   : 223
Monitor Only              : 58

Log file saved at: logs/collections_agent_log.csv

✅ Level 1 Basic Agent is complete and working!
This agent combines rule-based logic with simple reasoning and ethical guardrails.
